# scGPT RNA Embedding Fusion & Spatial Clustering (D1 Lymph Node)
This notebook integrates **scGPT** high-dimensional RNA embeddings with low-dimensional PCA features of RNA (omic1) and ADT/protein (omic2).
It builds a K-Nearest Neighbors (KNN) cell graph on the fused representation and performs clustering using KMeans and Leiden algorithms, evaluating their performance against manual cell type annotations.

### Kaggle Requirements:
1. **scGPT Model Dataset** (e.g., `scgpt-human` containing `scGPT_human` weight folder).
2. **Lymph Node Dataset** (e.g., `lymph-node-data` containing `adata_RNA.h5ad`, `adata_ADT.h5ad`, and `annotation.csv` for D1).
3. **GPU Accelerator** enabled (T4 or P100) for running scGPT embedding generation.

In [ ]:
# 1. Environment Setup (Kaggle & Colab compatible)
# Force install PyPI torch and torchtext compatible versions first
!pip install -q torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 torchtext==0.18.0

# Uninstall any conflicting numpy/scipy packages
!pip uninstall -y numpy scipy scikit-learn pandas scanpy anndata datasets
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy*
!rm -rf /usr/local/lib/python3.12/dist-packages/scipy*

# Install strictly compatible Spring 2024 Stack versions
!pip install -q "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.4.2" "pandas==2.2.2"

# Install omics and graph clustering packages
!pip install -q scanpy anndata datasets scikit-misc leidenalg

# Install scGPT library
!pip install -q scgpt==0.2.4 --no-deps

# Automatically restart the session/kernel to load the new numpy/scipy/pandas versions
import os
print("Restarting kernel to apply package installations. Please wait a moment, then run the next cells...")
os.kill(os.getpid(), 9)


In [ ]:
# 2. Imports and Seed Initialization
import os
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from torch.backends import cudnn
import sklearn
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.sparse import coo_matrix
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Optional
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def fix_seed(seed: int = 2026):
    """Fix random seed for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

fix_seed(2026)

In [ ]:
# 3. Data Loading
# Toggle directories automatically for Kaggle vs Local environments
KAGGLE_DATA_DIR = '/kaggle/input/datasets/sadmanbiazidarnob/lymph-node-data/Dataset11_Human_Lymph_Node_D1/'
LOCAL_DATA_DIR = 'data/10x_human_lymph_node_D1/'

if os.path.exists(KAGGLE_DATA_DIR):
    DATA_DIR = KAGGLE_DATA_DIR
elif os.path.exists(LOCAL_DATA_DIR):
    DATA_DIR = LOCAL_DATA_DIR
else:
    print("Data directories not found. Downloading dataset D1 locally...")
    os.makedirs('data', exist_ok=True)
    folder_id = '1U06il0AEEEryRXUVXESUEBWDY_0fH-s9'
    os.system(f"gdown --id {folder_id} --folder --output data")
    DATA_DIR = LOCAL_DATA_DIR

print(f"Loading data from {DATA_DIR}...")
adata_rna = sc.read_h5ad(os.path.join(DATA_DIR, 'adata_RNA.h5ad'))
adata_protein = sc.read_h5ad(os.path.join(DATA_DIR, 'adata_ADT.h5ad'))

# Load cell type annotations
annotation_path = os.path.join(DATA_DIR, 'annotation.csv')
if os.path.exists(annotation_path):
    annotation = pd.read_csv(annotation_path)
    if 'manual-anno' in annotation.columns:
        annotation = annotation.rename(columns={'manual-anno': 'ground_truth'})
    if 'Unnamed: 0' in annotation.columns:
        annotation = annotation.rename(columns={'Unnamed: 0': 'barcode'})
    elif 'Barcode' in annotation.columns:
        annotation = annotation.rename(columns={'Barcode': 'barcode'})
    annotation = annotation.set_index('barcode')
    
    adata_rna.obs = adata_rna.obs.join(annotation, how='left')
    adata_rna.obs['ground_truth'] = adata_rna.obs['ground_truth'].fillna('unknown')
    
    adata_protein.obs = adata_protein.obs.join(annotation, how='left')
    adata_protein.obs['ground_truth'] = adata_protein.obs['ground_truth'].fillna('unknown')
else:
    print("Warning: annotation.csv not found.")

print("RNA shape:", adata_rna.shape)
print("Protein/ADT shape:", adata_protein.shape)

In [ ]:
# 4. scGPT Embedding Generation
from scgpt.tasks.cell_emb import embed_data

KAGGLE_MODEL_DIR = '/kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human'
LOCAL_MODEL_DIR = 'scGPT_human'

if os.path.exists(KAGGLE_MODEL_DIR):
    model_dir = KAGGLE_MODEL_DIR
else:
    model_dir = LOCAL_MODEL_DIR

adata_rna.var_names_make_unique()
adata_protein.var_names_make_unique()
adata_rna.var['gene_names'] = adata_rna.var.index.str.upper()

print("Running scGPT embedding...")
if os.path.exists(model_dir):
    adata_emb = embed_data(
        adata_or_file=adata_rna.copy(), 
        model_dir=model_dir, 
        gene_col="gene_names", 
        max_length=1200, 
        batch_size=64, 
        obs_to_save=None, 
        device=device, 
        use_fast_transformer=False, 
        return_new_adata=False
    )
    X_scGPT = adata_emb.obsm["X_scGPT"]
    print("scGPT embedding generated successfully.")
else:
    print(f"Warning: Model path '{model_dir}' not found. Generating dummy random embeddings for local testing/validation.")
    X_scGPT = np.random.normal(size=(adata_rna.n_obs, 512))

adata_rna.obsm['X_scGPT'] = X_scGPT
print("scGPT matrix shape:", X_scGPT.shape)

In [ ]:
# 5. Preprocessing Omics Data (Omic1 & Omic2)
def pca(adata: ad.AnnData, use_reps=None, n_comps=10):
    """Perform PCA for dimensionality reduction."""
    pca_model = PCA(n_components=n_comps)
    data = adata.obsm[use_reps] if use_reps else adata.X
    data = data.toarray() if sp.issparse(data) else data
    return pca_model.fit_transform(data)

def clr_normalize_each_cell(adata: ad.AnnData, inplace=True):
    """Normalize count vector for each cell using CLR normalization."""
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x))
        return np.log1p(x / exp)

    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1, adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
    )
    return adata

print("Preprocessing Omic 1 (RNA)...")
sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)
sc.pp.scale(adata_rna)

adata_rna_high = adata_rna[:, adata_rna.var['highly_variable']]
n_comps_rna = adata_protein.n_vars - 1
adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=n_comps_rna)
print(f"RNA PCA feature representation shape: {adata_rna.obsm['feat'].shape}")

print("Preprocessing Omic 2 (ADT/Protein)...")
adata_protein = clr_normalize_each_cell(adata_protein)
sc.pp.scale(adata_protein)
n_comps_protein = adata_protein.n_vars - 1
adata_protein.obsm['feat'] = pca(adata_protein, n_comps=n_comps_protein)
print(f"Protein PCA feature representation shape: {adata_protein.obsm['feat'].shape}")

In [ ]:
# 6. Concatenate scGPT Embedding with Omic1 and Omic2
X_scGPT = adata_rna.obsm['X_scGPT']
feat_rna = adata_rna.obsm['feat']
feat_protein = adata_protein.obsm['feat']

# Fusing scGPT embedding matrix with RNA and ADT features
joint_feat = np.concatenate((X_scGPT, feat_rna, feat_protein), axis=1)
adata_rna.obsm['joint_feat'] = joint_feat
print("Concatenated (fused) feature matrix shape:", joint_feat.shape)

In [ ]:
# 7. KNN Graph Construction and Spatial Clustering
print("Constructing KNN graph on joint features...")
sc.pp.neighbors(adata_rna, use_rep='joint_feat', n_neighbors=10)
print("KNN graph computed successfully.")

# Determine target number of clusters from ground truth
valid_labels = adata_rna.obs['ground_truth'].dropna().unique()
target_labels = [l for l in valid_labels if l not in ['Exclude', 'unknown']]
n_clusters = len(target_labels) if len(target_labels) > 0 else 11
print(f"Target cluster count: {n_clusters}")

# 1. KMeans Clustering
print("Running KMeans...")
k_means_model = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
adata_rna.obs['kmeans_clusters'] = k_means_model.fit_predict(joint_feat).astype(str)

# 2. Leiden Clustering on the KNN Graph
print("Running Leiden resolution search...")
def search_res(adata, target_k, start=0.1, end=3.0, increment=0.05):
    for res in np.arange(start, end, increment):
        res = round(res, 3)
        sc.tl.leiden(adata, random_state=0, resolution=res, key_added='temp_leiden')
        unique_clusters = adata.obs['temp_leiden'].nunique()
        print(f"Resolution: {res} -> Clusters: {unique_clusters}")
        if unique_clusters == target_k:
            return res
    return 0.5

best_res = search_res(adata_rna, n_clusters)
sc.tl.leiden(adata_rna, random_state=0, resolution=best_res, key_added='leiden_clusters')
adata_rna.obs['leiden_clusters'] = adata_rna.obs['leiden_clusters'].astype(str)
print(f"Leiden clustering finished with resolution={best_res}")

In [ ]:
# 8. Cluster Performance Evaluation
def evaluate_clustering(y_true_series, y_pred_series, features_matrix, name=""):
    # Filter out Exclude/unknown values from verification metrics
    mask = (y_true_series != 'Exclude') & (y_true_series != 'unknown')
    y_true = y_true_series[mask].astype(str)
    y_pred = y_pred_series[mask].astype(str)
    feats = features_matrix[mask]
    
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    homogeneity = homogeneity_score(y_true, y_pred)
    v_measure = v_measure_score(y_true, y_pred)
    sil = silhouette_score(feats, y_pred.astype(int) if y_pred.str.isdigit().all() else pd.factorize(y_pred)[0])
    
    print(f"\n=== {name} Clustering Performance ===")
    print(f"ARI: {ari:.4f}")
    print(f"NMI: {nmi:.4f}")
    print(f"AMI: {ami:.4f}")
    print(f"Homogeneity: {homogeneity:.4f}")
    print(f"V-measure: {v_measure:.4f}")
    print(f"Silhouette: {sil:.4f}")
    return {"ARI": ari, "NMI": nmi, "AMI": ami, "Homogeneity": homogeneity, "V-measure": v_measure, "Silhouette": sil}

y_true = adata_rna.obs['ground_truth']
kmeans_res = evaluate_clustering(y_true, adata_rna.obs['kmeans_clusters'], joint_feat, "KMeans")
leiden_res = evaluate_clustering(y_true, adata_rna.obs['leiden_clusters'], joint_feat, "Leiden (KNN graph-based)")

In [ ]:
# 9. UMAP and Spatial Visualization
print("Computing UMAP projections...")
sc.tl.umap(adata_rna)

fig, axes = plt.subplots(3, 2, figsize=(15, 18))

# Plot Ground Truth
sc.pl.umap(adata_rna, color='ground_truth', ax=axes[0, 0], title='UMAP: Ground Truth', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='ground_truth', ax=axes[0, 1], title='Spatial: Ground Truth', show=False, size=25)

# Plot KMeans
sc.pl.umap(adata_rna, color='kmeans_clusters', ax=axes[1, 0], title='UMAP: KMeans Clusters', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='kmeans_clusters', ax=axes[1, 1], title='Spatial: KMeans Clusters', show=False, size=25)

# Plot Leiden
sc.pl.umap(adata_rna, color='leiden_clusters', ax=axes[2, 0], title='UMAP: Leiden Clusters (KNN-based)', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='leiden_clusters', ax=axes[2, 1], title='Spatial: Leiden Clusters (KNN-based)', show=False, size=25)

plt.tight_layout()
plt.show()